In [5]:
import json
import random
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import duckdb
import sys
import json
import time
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root:", PROJECT_ROOT)

SPIDER_DEV = Path("../data/spider/dev.json")

with open(SPIDER_DEV) as f:
    spider = json.load(f)

print("Total Spider questions:", len(spider))
print("Unique DBs:", len(set(s["db_id"] for s in spider)))
random.seed(42)
SAMPLE_SIZE = 100

eval_samples = random.sample(spider, SAMPLE_SIZE)

print("Evaluating mixed questions:", len(eval_samples))
print("DBs involved:", len(set(s["db_id"] for s in eval_samples)))
from agents.cartographer import (
    build_schema_graph,
    build_df,
    build_faiss_index,
    graphrag_cartographer
)

from agents.architect import architect_ensemble
from llama_cpp import Llama

MODELS_DIR = Path("../models")

llm_large = Llama(
    model_path=str(MODELS_DIR / "deepseek-coder-6.7b-instruct.Q4_K_M.gguf"),
    n_ctx=2048,
    n_threads=8,
    n_batch=256
)

llm_medium = Llama(
    model_path=str(MODELS_DIR / "mistral-7b-instruct-v0.2.Q4_K_M.gguf"),
    n_ctx=2048,
    n_threads=8,
    n_batch=256
)

print("✅ LLMs loaded")
def exec_result_safe(con, sql, timeout_sec=2.0, max_rows=10000):
    start = time.time()
    try:
        res = con.execute(sql).fetchall()

        if time.time() - start > timeout_sec:
            return "TIMEOUT"

        if len(res) > max_rows:
            return "TOO_MANY_ROWS"

        return res
    except Exception:
        return "ERROR"


def normalize_result(res):
    if not isinstance(res, list):
        return res
    return sorted(tuple(row) for row in res)


def results_match(pred, gold):
    if isinstance(pred, str) or isinstance(gold, str):
        return False
    return normalize_result(pred) == normalize_result(gold)


def classify_outcome(pred_res, gold_res):
    if pred_res == "TIMEOUT":
        return "timeout"
    if pred_res == "ERROR":
        return "execution_error"
    if pred_res == "TOO_MANY_ROWS":
        return "row_limit"
    if results_match(pred_res, gold_res):
        return "correct"
    return "wrong_logic"
SCHEMA_CACHE = {}

def load_schema_artifacts(db_id):
    if db_id in SCHEMA_CACHE:
        return SCHEMA_CACHE[db_id]

    db_path = Path(f"../data/spider/database/{db_id}/{db_id}.sqlite")

    graph, schema_texts, schema_ids, doc_tokens = build_schema_graph(db_path)
    df_stats = build_df(doc_tokens)
    embedder, faiss_index = build_faiss_index(schema_texts)

    artifacts = {
        "db_path": db_path,
        "graph": graph,
        "schema_texts": schema_texts,
        "schema_ids": schema_ids,
        "doc_tokens": doc_tokens,
        "df_stats": df_stats,
        "embedder": embedder,
        "faiss_index": faiss_index
    }

    SCHEMA_CACHE[db_id] = artifacts
    return artifacts
def evaluate_one(sample):
    question = sample["question"]
    gold_sql = sample["query"]
    db_id = sample["db_id"]

    artifacts = load_schema_artifacts(db_id)
    con = duckdb.connect(str(artifacts["db_path"]))

    gold_res = exec_result_safe(con, gold_sql)
    if isinstance(gold_res, str):
        return "gold_invalid", None, gold_sql, db_id

    # --- Cartographer ---
    _ = graphrag_cartographer(
        question=question,
        graph=artifacts["graph"],
        schema_texts=artifacts["schema_texts"],
        schema_ids=artifacts["schema_ids"],
        embedder=artifacts["embedder"],
        faiss_index=artifacts["faiss_index"],
        doc_tokens=artifacts["doc_tokens"],
        df=artifacts["df_stats"]
    )

    # --- Architect ---
    candidates = architect_ensemble(
        question=question,
        schema="\n".join(artifacts["schema_texts"]),
        llm_large=llm_large,
        llm_medium=llm_medium
    )

    for cand in candidates:
        sql = cand["sql"]
        pred_res = exec_result_safe(con, sql)
        outcome = classify_outcome(pred_res, gold_res)

        if outcome == "correct":
            return "correct", sql, gold_sql, db_id

    return "wrong_logic", None, gold_sql, db_id
records = []
correct = 0
start = time.time()

for i, sample in enumerate(eval_samples, 1):
    outcome, pred_sql, gold_sql, db_id = evaluate_one(sample)

    is_correct = (outcome == "correct")
    correct += int(is_correct)
    acc = correct / i

    print(f"\n[{i}] DB={db_id}")
    print("Q:", sample["question"])
    print("Pred:", pred_sql)
    print("Gold:", gold_sql)
    print("Outcome:", outcome.upper())
    print(f"Running accuracy: {acc:.2%}")

    records.append({
        "db_id": db_id,
        "question": sample["question"],
        "pred_sql": pred_sql,
        "gold_sql": gold_sql,
        "outcome": outcome,
        "correct": is_correct
    })

elapsed = time.time() - start
df = pd.DataFrame(records)

print("\n================ FINAL RESULTS ================")
print("Overall Accuracy:", df["correct"].mean())
print("Avg Time / Query:", elapsed / len(df))

print("\nOutcome Distribution:")
print(df["outcome"].value_counts())

print("\nPer-DB Accuracy (top 10):")
print(
    df.groupby("db_id")["correct"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)



✅ Project root: C:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4
Total Spider questions: 1034
Unique DBs: 20
Evaluating mixed questions: 100
DBs involved: 18


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 
AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


✅ LLMs loaded


Llama.generate: prefix-match hit



[1] DB=flight_2
Q: Give the code of the airport with the least flights.
Pred: SELECT a.AirportName, a.AirportCode FROM airports a JOIN (SELECT SourceAirport, COUNT(*) AS FlightCount FROM flights GROUP BY SourceAirport ORDER BY FlightCount ASC LIMIT 1 OFFSET ((SELECT COUNT(*) FROM airports) - 1)) f ON a.AirportCode = f.SourceAirport;
Gold: SELECT T1.AirportCode FROM AIRPORTS AS T1 JOIN FLIGHTS AS T2 ON T1.AirportCode  =  T2.DestAirport OR T1.AirportCode  =  T2.SourceAirport GROUP BY T1.AirportCode ORDER BY count(*) LIMIT 1
Outcome: CORRECT
Running accuracy: 100.00%


...


[97] DB=voter_1
Q: What are the create dates, states, and phone numbers of the votes that were for the contestant named 'Tabatha Gehling'?
Pred: None
Gold: SELECT T2.created ,  T2.state ,  T2.phone_number FROM contestants AS T1 JOIN votes AS T2 ON T1.contestant_number  =  T2.contestant_number WHERE T1.contestant_name  =  'Tabatha Gehling'
Outcome: WRONG_LOGIC
Running accuracy: 35.05%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



[98] DB=poker_player
Q: List the names of poker players ordered by the final tables made in ascending order.
Pred: None
Gold: SELECT T1.Name FROM people AS T1 JOIN poker_player AS T2 ON T1.People_ID  =  T2.People_ID ORDER BY T2.Final_Table_Made
Outcome: WRONG_LOGIC
Running accuracy: 34.69%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



[99] DB=pets_1
Q: Find the average weight for each pet type.
Pred: None
Gold: SELECT avg(weight) ,  pettype FROM pets GROUP BY pettype
Outcome: WRONG_LOGIC
Running accuracy: 34.34%

[100] DB=tvshow
Q: What is the TV Channel of TV series with Episode "A Love of a Lifetime"? List the TV Channel's series name.
Pred: None
Gold: SELECT T1.series_name FROM TV_Channel AS T1 JOIN TV_series AS T2 ON T1.id = T2.Channel WHERE T2.Episode = "A Love of a Lifetime";
Outcome: GOLD_INVALID
Running accuracy: 34.00%

================ FINAL RESULTS ================
Overall Accuracy: 0.34
Avg Time / Query: 36.32026398658753

Outcome Distribution:
outcome
correct         34
gold_invalid    34
wrong_logic     32
Name: count, dtype: int64

Per-DB Accuracy (top 10):
db_id
battle_death                    1.000000
employee_hire_evaluation        0.800000
orchestra                       0.800000
course_teach                    0.500000
dog_kennels                     0.500000
pets_1                          0.50